# Qwen2.5-0.5B Base 在 peS2o validation 上的 Perplexity

This notebook evaluates; it does not train or update the model.

默认评测 1,000 篇文档：320 篇 S2ORC 完整论文和 680 篇 S2AG 标题摘要。
它会将进度、最终 loss/perplexity 和结果 JSON 保存到 Weights & Biases。

运行前请在 Colab 选择 **Runtime → Change runtime type → V100 GPU**，然后依次运行所有单元格。


In [ ]:
# Install only the packages that Colab does not reliably preinstall.
# Equivalent command: python -m pip install transformers wandb
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "transformers>=4.46,<5",
    "wandb>=0.18,<1",
])
print("Dependencies installed. If Colab asks for a runtime restart, restart and run all cells again.")


In [ ]:
from __future__ import annotations

import gzip
import io
import json
import math
import time
import urllib.request
from pathlib import Path
from typing import Any, Callable, Iterable, Iterator


SUPPORTED_SOURCES = {"s2orc", "s2ag"}


def source_family(source: str) -> str:
    family = source.split("/", maxsplit=1)[0]
    if family not in SUPPORTED_SOURCES:
        raise ValueError(f"unsupported source {source}")
    return family


def validate_record(record: dict, expected_source: str | None = None) -> dict:
    if not isinstance(record, dict):
        raise ValueError("each record must be a JSON object")
    for key in ("id", "source", "text"):
        if not isinstance(record.get(key), str) or not record[key]:
            raise ValueError(f"record field {key!r} must be a non-empty string")
    record_source = source_family(record["source"])
    if expected_source is not None and record_source != source_family(expected_source):
        raise ValueError(
            f"expected source {expected_source}, received {record['source']}"
        )
    return record


def iter_jsonl_gz(url: str) -> Iterator[dict]:
    with urllib.request.urlopen(url, timeout=300) as response:
        with gzip.GzipFile(fileobj=response) as compressed:
            with io.TextIOWrapper(compressed, encoding="utf-8") as text_stream:
                for line_number, line in enumerate(text_stream, start=1):
                    if not line.strip():
                        continue
                    try:
                        record = json.loads(line)
                    except json.JSONDecodeError as error:
                        raise ValueError(
                            f"invalid JSON at line {line_number} from {url}"
                        ) from error
                    yield validate_record(record)


def collect_source_records(
    urls: Iterable[str], limits: dict[str, int | None]
) -> dict[str, list[dict]]:
    unknown_sources = set(limits) - SUPPORTED_SOURCES
    if unknown_sources:
        raise ValueError(f"unsupported requested sources: {sorted(unknown_sources)}")
    for source, limit in limits.items():
        if limit is not None and limit < 1:
            raise ValueError(f"limit for {source} must be positive or None")

    records = {source: [] for source in limits}

    def all_finite_limits_reached() -> bool:
        return all(
            limit is not None and len(records[source]) >= limit
            for source, limit in limits.items()
        )

    for url in urls:
        for record in iter_jsonl_gz(url):
            source = source_family(record["source"])
            if source not in records:
                continue
            limit = limits[source]
            if limit is None or len(records[source]) < limit:
                records[source].append(record)
            if all_finite_limits_reached():
                return records

    short = {
        source: {"expected": limit, "actual": len(records[source])}
        for source, limit in limits.items()
        if limit is not None and len(records[source]) < limit
    }
    if short:
        raise ValueError(f"validation streams ended before limits were met: {short}")
    return records


def iter_packed_sequences(
    records: Iterable[dict], tokenizer: Any, sequence_length: int
) -> Iterator[dict]:
    if sequence_length < 2:
        raise ValueError("sequence_length must be at least 2")
    eos_token_id = tokenizer.eos_token_id
    if eos_token_id is None:
        raise ValueError("tokenizer must define eos_token_id")

    token_buffer: list[int] = []
    origin_buffer: list[str] = []

    for raw_record in records:
        record = validate_record(raw_record)
        document_id = record["id"]
        token_ids = tokenizer.encode(record["text"], add_special_tokens=False)
        token_ids.append(eos_token_id)

        position = 0
        while position < len(token_ids):
            space = sequence_length - len(token_buffer)
            next_position = min(position + space, len(token_ids))
            piece = token_ids[position:next_position]
            token_buffer.extend(piece)
            origin_buffer.extend([document_id] * len(piece))
            position = next_position

            if len(token_buffer) == sequence_length:
                yield {
                    "input_ids": token_buffer,
                    "labels": token_buffer.copy(),
                    "document_ids": list(dict.fromkeys(origin_buffer)),
                }
                token_buffer = []
                origin_buffer = []

    if token_buffer:
        padding = sequence_length - len(token_buffer)
        yield {
            "input_ids": token_buffer + [eos_token_id] * padding,
            "labels": token_buffer + [-100] * padding,
            "document_ids": list(dict.fromkeys(origin_buffer)),
        }


def perplexity_from_loss(mean_loss: float) -> float:
    if not math.isfinite(mean_loss):
        raise ValueError("mean loss must be finite")
    try:
        return math.exp(mean_loss)
    except OverflowError:
        return float("inf")


def combine_source_metrics(metrics: Iterable[dict]) -> dict:
    metrics = list(metrics)
    total_tokens = sum(int(item["predicted_tokens"]) for item in metrics)
    if total_tokens <= 0:
        raise ValueError("predicted token total must be positive")
    total_nll = sum(float(item["negative_log_likelihood"]) for item in metrics)
    loss = total_nll / total_tokens
    return {
        "negative_log_likelihood": total_nll,
        "predicted_tokens": total_tokens,
        "loss": loss,
        "perplexity": perplexity_from_loss(loss),
    }


def count_shifted_targets(labels: Any) -> int:
    return int(labels[:, 1:].ne(-100).sum().item())


def _iter_batches(items: Iterable[dict], batch_size: int) -> Iterator[list[dict]]:
    if batch_size < 1:
        raise ValueError("batch_size must be positive")
    batch: list[dict] = []
    for item in items:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch


def evaluate_source(
    model: Any,
    packed_sequences: Iterable[dict],
    source: str,
    batch_size: int,
    device: str,
    log_every_steps: int,
    progress_callback: Callable[[dict], None] | None = None,
) -> dict:
    import torch

    if source not in SUPPORTED_SOURCES:
        raise ValueError(f"unsupported source {source}")
    if log_every_steps < 1:
        raise ValueError("log_every_steps must be positive")

    start = time.perf_counter()
    total_nll = 0.0
    predicted_tokens = 0
    input_tokens = 0
    sequence_count = 0
    batch_count = 0
    document_ids: list[str] = []
    seen_document_ids: set[str] = set()
    last_logged_batch = 0

    try:
        with torch.inference_mode():
            for batch_count, batch in enumerate(
                _iter_batches(packed_sequences, batch_size), start=1
            ):
                input_ids = torch.tensor(
                    [item["input_ids"] for item in batch],
                    dtype=torch.long,
                    device=device,
                )
                labels = torch.tensor(
                    [item["labels"] for item in batch],
                    dtype=torch.long,
                    device=device,
                )
                attention_mask = labels.ne(-100).long()
                batch_targets = count_shifted_targets(labels)
                if batch_targets < 1:
                    raise ValueError("a packed batch contained no prediction targets")

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                )
                batch_loss = float(outputs.loss.item())
                if not math.isfinite(batch_loss):
                    raise ValueError(
                        f"non-finite loss for {source} at batch {batch_count}"
                    )

                total_nll += batch_loss * batch_targets
                predicted_tokens += batch_targets
                input_tokens += int(labels.ne(-100).sum().item())
                sequence_count += len(batch)
                for item in batch:
                    for document_id in item["document_ids"]:
                        if document_id not in seen_document_ids:
                            seen_document_ids.add(document_id)
                            document_ids.append(document_id)

                if progress_callback is not None and (
                    batch_count == 1 or batch_count % log_every_steps == 0
                ):
                    elapsed = max(time.perf_counter() - start, 1e-9)
                    running_loss = total_nll / predicted_tokens
                    progress_callback(
                        {
                            "source": source,
                            "batch": batch_count,
                            "sequences": sequence_count,
                            "predicted_tokens": predicted_tokens,
                            "loss": running_loss,
                            "perplexity": perplexity_from_loss(running_loss),
                            "tokens_per_second": predicted_tokens / elapsed,
                            "elapsed_seconds": elapsed,
                            "gpu_memory_gb": torch.cuda.memory_allocated() / (1024**3),
                        }
                    )
                    last_logged_batch = batch_count
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError(
            f"CUDA ran out of memory with batch_size={batch_size}; "
            "reduce CONFIG['batch_size'] and rerun"
        ) from error

    if predicted_tokens < 1:
        raise ValueError(f"no prediction targets were evaluated for {source}")

    elapsed = max(time.perf_counter() - start, 1e-9)
    loss = total_nll / predicted_tokens
    result = {
        "source": source,
        "documents": len(document_ids),
        "document_ids": document_ids,
        "sequences": sequence_count,
        "batches": batch_count,
        "input_tokens": input_tokens,
        "predicted_tokens": predicted_tokens,
        "negative_log_likelihood": total_nll,
        "loss": loss,
        "perplexity": perplexity_from_loss(loss),
        "elapsed_seconds": elapsed,
        "tokens_per_second": predicted_tokens / elapsed,
    }

    if progress_callback is not None and last_logged_batch != batch_count:
        progress_callback({**result, "batch": batch_count})
    return result


def write_result_json(path: str | Path, result: dict) -> Path:
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        json.dumps(result, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False)
        + "\n",
        encoding="utf-8",
    )
    return output_path


In [ ]:
import platform
import random
import sys
from pathlib import Path

import torch
import transformers
import wandb
from transformers import AutoModelForCausalLM, AutoTokenizer


CONFIG = {
    "model_id": "Qwen/Qwen2.5-0.5B",
    "sequence_length": 2048,
    "batch_size": 4,
    "seed": 42,
    "log_every_steps": 10,
    "wandb_project": "lshbloom-pes2o",
    "run_name": "qwen2.5-0.5b-base-pes2o-valid-1000",
    "s2orc_documents": 320,
    "s2ag_documents": 680,
}

VALIDATION_URLS = [
    "https://huggingface.co/datasets/allenai/peS2o/resolve/main/data/v2/validation-00000-of-00002.json.gz",
    "https://huggingface.co/datasets/allenai/peS2o/resolve/main/data/v2/validation-00001-of-00002.json.gz",
]
OUTPUT_PATH = Path("/content/results/qwen2.5-0.5b-base-pes2o-valid-1000.json")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. In Colab, select Runtime > Change runtime type > GPU.")

random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])

print("GPU:", torch.cuda.get_device_name(0))
print("Configuration:", json.dumps(CONFIG, indent=2))


## 读取固定的 1,000 篇 validation 文档

代码检查每条记录的 `source` 字段，并在两个官方 validation 文件中收集到 320 篇 S2ORC 和 680 篇 S2AG 后停止。


In [ ]:
source_limits = {
    "s2orc": CONFIG["s2orc_documents"],
    "s2ag": CONFIG["s2ag_documents"],
}
records_by_source = collect_source_records(VALIDATION_URLS, source_limits)

actual_counts = {
    source: len(records) for source, records in records_by_source.items()
}
for source, limit in source_limits.items():
    if limit is not None and actual_counts[source] != limit:
        raise RuntimeError(
            f"Unexpected sample count for {source}: "
            f"expected {limit}, received {actual_counts[source]}"
        )

print("Loaded documents:", actual_counts)
print("First IDs:", {source: records[0]["id"] for source, records in records_by_source.items()})


## 加载模型并评测

`wandb.login()` 会显示登录入口。模型以 FP16 加载到 V100，只执行前向计算。首先运行一个很小的 smoke test，成功后再计算 1,000 篇文档。


In [ ]:
wandb.login()
run = wandb.init(
    project=CONFIG["wandb_project"],
    name=CONFIG["run_name"],
    config={
        **CONFIG,
        "dataset": "allenai/peS2o",
        "dataset_version": "v2",
        "validation_urls": VALIDATION_URLS,
        "gpu": torch.cuda.get_device_name(0),
    },
)


def log_progress(metrics):
    prefix = f"eval/{metrics['source']}"
    payload = {
        f"{prefix}/batch": metrics["batch"],
        f"{prefix}/sequences": metrics["sequences"],
        f"{prefix}/predicted_tokens": metrics["predicted_tokens"],
        f"{prefix}/running_loss": metrics["loss"],
        f"{prefix}/running_perplexity": metrics["perplexity"],
        f"{prefix}/tokens_per_second": metrics["tokens_per_second"],
        f"{prefix}/elapsed_seconds": metrics["elapsed_seconds"],
    }
    if "gpu_memory_gb" in metrics:
        payload[f"{prefix}/gpu_memory_gb"] = metrics["gpu_memory_gb"]
    wandb.log(payload)


try:
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_id"])
    if tokenizer.eos_token_id is None:
        raise RuntimeError("Qwen tokenizer does not define an EOS token")

    model = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_id"],
        torch_dtype=torch.float16,
    ).to("cuda")
    model.eval()

    smoke_length = min(128, CONFIG["sequence_length"])
    smoke_sequence = next(
        iter_packed_sequences(records_by_source["s2ag"][:1], tokenizer, smoke_length)
    )
    smoke_input_ids = torch.tensor(
        [smoke_sequence["input_ids"]], dtype=torch.long, device="cuda"
    )
    smoke_labels = torch.tensor(
        [smoke_sequence["labels"]], dtype=torch.long, device="cuda"
    )
    smoke_attention_mask = smoke_labels.ne(-100).long()
    smoke_targets = count_shifted_targets(smoke_labels)
    if smoke_targets < 1:
        raise RuntimeError("Smoke test produced no prediction targets")

    with torch.inference_mode():
        smoke_output = model(
            input_ids=smoke_input_ids,
            attention_mask=smoke_attention_mask,
            labels=smoke_labels,
        )
    smoke_loss = float(smoke_output.loss.item())
    if not math.isfinite(smoke_loss):
        raise RuntimeError(f"Smoke test produced non-finite loss: {smoke_loss}")
    wandb.log({
        "smoke_test/passed": 1,
        "smoke_test/loss": smoke_loss,
        "smoke_test/predicted_tokens": smoke_targets,
    })
    print(f"Smoke test passed: loss={smoke_loss:.4f}")

    del smoke_input_ids, smoke_labels, smoke_attention_mask, smoke_output
    torch.cuda.empty_cache()

    source_results = {}
    for source in ("s2orc", "s2ag"):
        print(f"Evaluating {source}: {len(records_by_source[source])} documents")
        packed_sequences = iter_packed_sequences(
            records_by_source[source], tokenizer, CONFIG["sequence_length"]
        )
        source_results[source] = evaluate_source(
            model=model,
            packed_sequences=packed_sequences,
            source=source,
            batch_size=CONFIG["batch_size"],
            device="cuda",
            log_every_steps=CONFIG["log_every_steps"],
            progress_callback=log_progress,
        )
        print(
            f"{source}: loss={source_results[source]['loss']:.4f}, "
            f"perplexity={source_results[source]['perplexity']:.4f}"
        )

    overall = combine_source_metrics(source_results.values())
    if not math.isfinite(overall["perplexity"]):
        raise RuntimeError(f"Overall perplexity is not finite: {overall['perplexity']}")

    result = {
        "config": CONFIG,
        "dataset": {
            "name": "allenai/peS2o",
            "version": "v2",
            "split": "validation",
            "urls": VALIDATION_URLS,
        },
        "environment": {
            "python": sys.version,
            "platform": platform.platform(),
            "torch": torch.__version__,
            "transformers": transformers.__version__,
            "wandb": wandb.__version__,
            "gpu": torch.cuda.get_device_name(0),
        },
        "sources": source_results,
        "overall": overall,
        "wandb_run_id": run.id,
        "wandb_run_url": run.url,
    }
    result_path = write_result_json(OUTPUT_PATH, result)

    for source, metrics in source_results.items():
        for key in ("documents", "input_tokens", "predicted_tokens", "loss", "perplexity", "tokens_per_second"):
            run.summary[f"final/{source}/{key}"] = metrics[key]
    for key in ("predicted_tokens", "loss", "perplexity"):
        run.summary[f"final/overall/{key}"] = overall[key]

    artifact = wandb.Artifact(
        name=f"{CONFIG['run_name']}-results-{run.id}",
        type="evaluation-results",
        description="Qwen2.5-0.5B Base perplexity on the peS2o V2 validation pilot",
    )
    artifact.add_file(str(result_path))
    run.log_artifact(artifact)

    print("\nFinal results")
    print("-------------")
    for source, metrics in source_results.items():
        print(
            f"{source:5s} | documents={metrics['documents']:4d} | "
            f"tokens={metrics['predicted_tokens']:9d} | "
            f"loss={metrics['loss']:.4f} | ppl={metrics['perplexity']:.4f}"
        )
    print(
        f"overall | tokens={overall['predicted_tokens']:9d} | "
        f"loss={overall['loss']:.4f} | ppl={overall['perplexity']:.4f}"
    )
    print("Result JSON:", result_path)
    print("W&B run:", run.url)
finally:
    wandb.finish()
